# subword model

## Neural Machine Translation of Rare Words with Subword Units
데이터 압축으로 쓰이던 bpe를 자연어에 쓴 논문이다.
단어보다 작은 subword unit을 사용하여 음운론적이고 형태학적으로 번역함으로써, open-vocabulary NMT모델을 소개한다.  


## Introduction
agglutination and compounding 단계가 포합된 언어의 경우, word level에서 하위 수준으로 내려가는 메커니즘이 필요하다.
- subword를 통한 open-vocabulary은 back-off dictionay나 큰 large vocabularies보다 효과적이다.
- byte pair encoding (BPE)(Gage, 1994)은 가변 길이의 문자열을 고정크기의 어휘를 통해 표현 할수 있기 때문에 인공신경망에서 효과적인 방법이다.

## Neural Machine Translation
모델은 Bi-direction GRU 유닛 사용
subword로 정보를 투영할 때의 장점
- named entities
- cognates and loanwords(의미 분화 및 외래어)
- morphologically complex words(복합어)

## Bytr Pair Encoding (BPE)
알고리즘은 모든 쌍을 인덱싱하고, 데이터 구조(dictionary)를 점진적으로 업데이트 한다.
예를 들어 ('A', 'B')안에서 모든 기호 쌍을 반복적으로 계산해서 가장 빈도가 높은 쌍을 새로운 기호 'AB'로 교체한다.
허프만 인코딩 처럼 하위 단어를 기반으로 새로운 단어를 번역하고 새성한다.

In [14]:
import re, collections
def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] +=freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)'+bigram+r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

vocab ={'l o w </w>' : 5, 'l o w e r </w>' : 2,
       'n e w e s t </w>' :6, 'w i d e s t </w>': 3}
num_merges = 10
for i in range(num_merges):
    pairs = get_stats(vocab)
    best = max(pairs, key=pairs.get) 
    vocab = merge_vocab(best, vocab) 
    print(best)
    
    


('e', 's')
('es', 't')
('est', '</w>')
('l', 'o')
('lo', 'w')
('n', 'e')
('ne', 'w')
('new', 'est</w>')
('low', '</w>')
('w', 'i')


여기서 lower는 low, er로 분할된다.

이 논문의 task인 translation에서 bpe의 사용은 2가지가 있다.


- source 어휘와 target 각각의 어휘
- source와 target의 어휘를 합침(joint BPE) 

전자는 어휘 사이즈와, 단어가 더 compact하고 후자는 source와 target segmentation 사이에 일관성이 있다.

## result
<img src="https://github.com/indexxlim/indexxlim.github.io/blob/main/diary.py/machine_learning/paper/images/subword/1_result.png?raw=true" itemprop="image" width="60%">

dict_items([('low</w>', 5), ('low e r </w>', 2), ('newest</w>', 6), ('wi d est</w>', 3)])

{'l o w </w>': 5,
 'l o w e r </w>': 2,
 'n e w e s t </w>': 6,
 'w i d e s t </w>': 3}